# California Real Estate Price Prediction

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}' if abs(x) > 1 else f'{x:.4f}')

print("✓ All libraries imported successfully!")
print(f"✓ XGBoost version: {xgb.__version__}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")

In [ ]:
# doad data 
data_path = 'CRMLSSold202401_filled.csv' 

print("Loading data...")
df_raw = pd.read_csv(data_path)

df_raw['CloseDate'] = pd.to_datetime(df_raw['CloseDate'])

print(f"\n{'='*80}")
print("DATA LOADED SUCCESSFULLY")
print(f"{'='*80}")
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Date range: {df_raw['CloseDate'].min().date()} to {df_raw['CloseDate'].max().date()}")
print(f"\nFirst few rows:")
df_raw.head()

In [ ]:

print("Property Types:")
print(df_raw['PropertyType'].value_counts())

print("\nProperty SubTypes (top 10):")
print(df_raw['PropertySubType'].value_counts().head(10))

In [ ]:
print(f"{'='*80}")
print("DATA FILTERING")
print(f"{'='*80}")

df = df_raw.copy()
initial_count = len(df)
print(f"\nStarting records: {initial_count:,}")

df = df[df['PropertyType'] == 'Residential'].copy()
print(f"After PropertyType filter: {len(df):,} ({initial_count - len(df):,} removed)")

count_before = len(df)
df = df[df['PropertySubType'] == 'SingleFamilyResidence'].copy()
print(f"After PropertySubType filter: {len(df):,} ({count_before - len(df):,} removed)")

print(f"\nRemoving INVALID transactions:")

count_before = len(df)
df = df[df['ClosePrice'] > 0].copy()
print(f"  ClosePrice <= 0: {count_before - len(df):,} removed")

count_before = len(df)
df = df[df['LivingArea'] > 0].copy()
print(f"  LivingArea <= 0: {count_before - len(df):,} removed")

count_before = len(df)
df = df[df['Latitude'].notna() & df['Longitude'].notna()].copy()
print(f"  Missing coordinates: {count_before - len(df):,} removed")

count_before = len(df)
df = df[(df['Latitude'] >= 32.5) & (df['Latitude'] <= 42.0)].copy()
df = df[(df['Longitude'] >= -124.5) & (df['Longitude'] <= -114.0)].copy()
print(f"  Invalid coordinates: {count_before - len(df):,} removed")

print(f"\n{'='*80}")
print(f"FINAL FILTERED DATASET: {len(df):,} rows")
print(f"Total removed: {initial_count - len(df):,} ({100*(initial_count - len(df))/initial_count:.1f}%)")
print(f"{'='*80}")

print(f"Price Statistics:")
print(f"  Min:    ${df['ClosePrice'].min():>15,.0f}")
print(f"  25th:   ${df['ClosePrice'].quantile(0.25):>15,.0f}")
print(f"  Median: ${df['ClosePrice'].median():>15,.0f}")
print(f"  Mean:   ${df['ClosePrice'].mean():>15,.0f}")
print(f"  75th:   ${df['ClosePrice'].quantile(0.75):>15,.0f}")
print(f"  Max:    ${df['ClosePrice'].max():>15,.0f}")

df_filtered = df.copy()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_filtered['ClosePrice'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Close Price ($)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Price Distribution', fontsize=14, fontweight='bold')
axes[0].axvline(df_filtered['ClosePrice'].median(), color='red', 
                linestyle='--', linewidth=2, label=f"Median: ${df_filtered['ClosePrice'].median():,.0f}")
axes[0].legend()

axes[1].hist(np.log10(df_filtered['ClosePrice']), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('log10(Close Price)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Price Distribution (Log Scale)', fontsize=14, fontweight='bold')

axes[2].boxplot(df_filtered['ClosePrice'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[2].set_ylabel('Close Price ($)', fontsize=12)
axes[2].set_title('Price Box Plot', fontsize=14, fontweight='bold')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter = axes[0].scatter(df_filtered['Longitude'], df_filtered['Latitude'], 
                         c=np.log10(df_filtered['ClosePrice']), cmap='YlOrRd', 
                         alpha=0.6, s=15, edgecolor='none')
axes[0].set_xlabel('Longitude', fontsize=12)
axes[0].set_ylabel('Latitude', fontsize=12)
axes[0].set_title('Geographic Distribution (colored by log price)', fontsize=14, fontweight='bold')
cbar = plt.colorbar(scatter, ax=axes[0])
cbar.set_label('log10(Price)', fontsize=11)

# Hexbin plot
hexbin = axes[1].hexbin(df_filtered['Longitude'], df_filtered['Latitude'], 
                        C=df_filtered['ClosePrice'], gridsize=30, cmap='viridis', 
                        reduce_C_function=np.median, mincnt=1)
axes[1].set_xlabel('Longitude', fontsize=12)
axes[1].set_ylabel('Latitude', fontsize=12)
axes[1].set_title('Median Price by Location', fontsize=14, fontweight='bold')
cbar2 = plt.colorbar(hexbin, ax=axes[1])
cbar2.set_label('Median Price ($)', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
print(f"{'='*80}")
print("TRAIN / TEST SPLIT")
print(f"{'='*80}")

df = df_filtered.copy()

min_date = df['CloseDate'].min()
max_date = df['CloseDate'].max()
date_span_days = (max_date - min_date).days

print(f"\nDate range: {min_date.date()} to {max_date.date()}")
print(f"Total span: {date_span_days} days")

# Determine split strategy
if date_span_days >= 180:  # 6+ months
    print(f"\n✓ Using STANDARD split: Last month as test set")
    last_month_start = max_date.replace(day=1)
    df_test = df[df['CloseDate'] >= last_month_start].copy()
    train_end = last_month_start - timedelta(days=1)
    train_start = last_month_start - pd.DateOffset(months=6)
    df_train = df[(df['CloseDate'] >= train_start) & (df['CloseDate'] <= train_end)].copy()
else:
    print(f"\n⚠️  Only {date_span_days} days available (<180)")
    print(f"✓ Using ALTERNATIVE split: 80/20 temporal split")
    cutoff_date = min_date + timedelta(days=int(date_span_days * 0.8))
    df_test = df[df['CloseDate'] >= cutoff_date].copy()
    df_train = df[df['CloseDate'] < cutoff_date].copy()

print(f"\nTraining Set:")
print(f"  Dates: {df_train['CloseDate'].min().date()} to {df_train['CloseDate'].max().date()}")
print(f"  Records: {len(df_train):,}")
print(f"  Price range: ${df_train['ClosePrice'].min():,.0f} - ${df_train['ClosePrice'].max():,.0f}")

print(f"\nTest Set:")
print(f"  Dates: {df_test['CloseDate'].min().date()} to {df_test['CloseDate'].max().date()}")
print(f"  Records: {len(df_test):,}")
print(f"  Price range: ${df_test['ClosePrice'].min():,.0f} - ${df_test['ClosePrice'].max():,.0f}")

# Create evaluation set (trimmed 0.5% on each end)
lower_pct = df_test['ClosePrice'].quantile(0.005)
upper_pct = df_test['ClosePrice'].quantile(0.995)
df_test_eval = df_test[(df_test['ClosePrice'] >= lower_pct) & 
                       (df_test['ClosePrice'] <= upper_pct)].copy()

print(f"\nEvaluation Set (0.5% trimmed):")
print(f"  Records: {len(df_test_eval):,} (removed {len(df_test) - len(df_test_eval)})")
print(f"  Price range: ${df_test_eval['ClosePrice'].min():,.0f} - ${df_test_eval['ClosePrice'].max():,.0f}")

In [ ]:
def create_features(df):
    """
    Create engineered features for XGBoost model.
    Returns DataFrame with 33 features.
    """
    df_feat = pd.DataFrame()
    
    df_feat['Latitude'] = df['Latitude']
    df_feat['Longitude'] = df['Longitude']
    df_feat['LivingArea'] = df['LivingArea']
    
    df_feat['BedroomsTotal'] = df['BedroomsTotal'].fillna(df['BedroomsTotal'].median())
    df_feat['BathroomsTotalInteger'] = df['BathroomsTotalInteger'].fillna(df['BathroomsTotalInteger'].median())
    
    if 'YearBuilt' in df.columns:
        df_feat['YearBuilt'] = df['YearBuilt'].fillna(df['YearBuilt'].median())
        df_feat['PropertyAge'] = 2024 - df_feat['YearBuilt']
        df_feat['PropertyAge_Squared'] = df_feat['PropertyAge'] ** 2
    
    if 'LotSizeSquareFeet' in df.columns:
        df_feat['LotSizeSquareFeet'] = df['LotSizeSquareFeet'].fillna(df['LotSizeSquareFeet'].median())
        df_feat['LotSize_Log'] = np.log1p(df_feat['LotSizeSquareFeet'])
    
    if 'GarageSpaces' in df.columns:
        df_feat['GarageSpaces'] = df['GarageSpaces'].fillna(0)
    if 'ParkingTotal' in df.columns:
        df_feat['ParkingTotal'] = df['ParkingTotal'].fillna(0)
    
    if 'PoolPrivateYN' in df.columns:
        df_feat['HasPool'] = (df['PoolPrivateYN'] == True).astype(int)
    
    if 'FireplacesTotal' in df.columns:
        df_feat['FireplacesTotal'] = df['FireplacesTotal'].fillna(0)
        df_feat['HasFireplace'] = (df_feat['FireplacesTotal'] > 0).astype(int)
    
    if 'Stories' in df.columns:
        stories_map = {'One': 1, 'Two': 2, 'Three': 3, 'ThreeOrMore': 3.5, 
                       'Four': 4, 'FourOrMore': 4.5}
        df_feat['Stories'] = df['Stories'].map(stories_map).fillna(1)
    
    df_feat['Lat_Lon_Interaction'] = df_feat['Latitude'] * df_feat['Longitude']
    df_feat['Lat_Squared'] = df_feat['Latitude'] ** 2
    df_feat['Lon_Squared'] = df_feat['Longitude'] ** 2
    
    # Distance to major cities
    # Los Angeles: 34.0522°N, -118.2437°W
    # San Francisco: 37.7749°N, -122.4194°W
    # San Diego: 32.7157°N, -117.1611°W
    
    df_feat['Dist_LA'] = np.sqrt(
        (df_feat['Latitude'] - 34.0522)**2 + 
        (df_feat['Longitude'] - (-118.2437))**2
    )
    df_feat['Dist_SF'] = np.sqrt(
        (df_feat['Latitude'] - 37.7749)**2 + 
        (df_feat['Longitude'] - (-122.4194))**2
    )
    df_feat['Dist_SD'] = np.sqrt(
        (df_feat['Latitude'] - 32.7157)**2 + 
        (df_feat['Longitude'] - (-117.1611))**2
    )
    df_feat['Min_Dist_Major_City'] = df_feat[['Dist_LA', 'Dist_SF', 'Dist_SD']].min(axis=1)
    
    df_feat['LivingArea_Log'] = np.log1p(df_feat['LivingArea'])
    df_feat['LivingArea_Squared'] = df_feat['LivingArea'] ** 2
    df_feat['LivingArea_Sqrt'] = np.sqrt(df_feat['LivingArea'])
    
    df_feat['Bed_per_sqft'] = df_feat['BedroomsTotal'] / (df_feat['LivingArea'] + 1)
    df_feat['Bath_per_sqft'] = df_feat['BathroomsTotalInteger'] / (df_feat['LivingArea'] + 1)
    df_feat['Total_Rooms'] = df_feat['BedroomsTotal'] + df_feat['BathroomsTotalInteger']
    
    df_feat['Coastal_Proximity'] = np.abs(df_feat['Longitude'] + 120)
    
    df_feat['Is_Northern_CA'] = (df_feat['Latitude'] > 37).astype(int)
    df_feat['Is_Southern_CA'] = (df_feat['Latitude'] < 35).astype(int)
    df_feat['Is_Central_CA'] = ((df_feat['Latitude'] >= 35) & (df_feat['Latitude'] <= 37)).astype(int)
    
    return df_feat

print(f"{'='*80}")
print("FEATURE ENGINEERING")
print(f"{'='*80}")

print("\nCreating features...")
X_train = create_features(df_train)
y_train = df_train['ClosePrice'].values

X_test = create_features(df_test)
y_test = df_test['ClosePrice'].values

X_test_eval = create_features(df_test_eval)
y_test_eval = df_test_eval['ClosePrice'].values

X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median()) 
X_test_eval = X_test_eval.fillna(X_train.median())

print(f"\n✓ Features created: {X_train.shape[1]}")
print(f"✓ Training samples: {X_train.shape[0]:,}")
print(f"✓ Test samples: {X_test.shape[0]:,}")
print(f"✓ Evaluation samples: {X_test_eval.shape[0]:,}")

print(f"\nFeature List:")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
print(f"{'='*80}")
print("TRAINING XGBOOST MODEL")
print(f"{'='*80}")

xgb_model = XGBRegressor(
    n_estimators=200,          
    max_depth=6,                
    learning_rate=0.1,         
    min_child_weight=3,        
    subsample=0.8,              
    colsample_bytree=0.8,       
    objective='reg:squarederror',  
    eval_metric='rmse',        
    random_state=42,            
    n_jobs=-1,                 
    verbosity=1                
)

print("\nModel Configuration:")
print(f"  n_estimators: {xgb_model.n_estimators}")
print(f"  max_depth: {xgb_model.max_depth}")
print(f"  learning_rate: {xgb_model.learning_rate}")
print(f"  subsample: {xgb_model.subsample}")
print(f"  colsample_bytree: {xgb_model.colsample_bytree}")

print("\nTraining model...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test_eval, y_test_eval)],
    verbose=20 
)

print("\n✓ Model trained successfully!")

In [ ]:
results = xgb_model.evals_result()

fig, ax = plt.subplots(figsize=(12, 6))

epochs = len(results['validation_0']['rmse'])
x_axis = range(0, epochs)

ax.plot(x_axis, results['validation_0']['rmse'], label='Train', linewidth=2)
ax.plot(x_axis, results['validation_1']['rmse'], label='Test', linewidth=2)
ax.legend(fontsize=12)
ax.set_xlabel('Boosting Round', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title('XGBoost Training History', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final Training RMSE: ${results['validation_0']['rmse'][-1]:,.0f}")
print(f"Final Test RMSE: ${results['validation_1']['rmse'][-1]:,.0f}")

In [ ]:
def calculate_mdape(y_true, y_pred):
    """Calculate Median Absolute Percent Error."""
    ape = np.abs((y_true - y_pred) / y_true) * 100
    return np.median(ape)

y_pred_train = xgb_model.predict(X_train)
y_pred_test = xgb_model.predict(X_test)
y_pred_eval = xgb_model.predict(X_test_eval)

r2_train = r2_score(y_train, y_pred_train)
mdape_train = calculate_mdape(y_train, y_pred_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
mae_train = mean_absolute_error(y_train, y_pred_train)

r2_eval = r2_score(y_test_eval, y_pred_eval)
mdape_eval = calculate_mdape(y_test_eval, y_pred_eval)
rmse_eval = np.sqrt(mean_squared_error(y_test_eval, y_pred_eval))
mae_eval = mean_absolute_error(y_test_eval, y_pred_eval)

print(f"{'='*80}")
print("MODEL EVALUATION RESULTS")
print(f"{'='*80}")

print(f"\nTraining Set Performance:")
print(f"  R²:     {r2_train:>10.4f}  ({r2_train*100:.2f}% variance explained)")
print(f"  MdAPE:  {mdape_train:>10.2f}%")
print(f"  RMSE:   ${rmse_train:>10,.0f}")
print(f"  MAE:    ${mae_train:>10,.0f}")

print(f"\n{'='*80}")
print(f"Test Set Performance (0.5% trimmed):")
print(f"  R²:     {r2_eval:>10.4f}  ({r2_eval*100:.2f}% variance explained)")
print(f"  MdAPE:  {mdape_eval:>10.2f}%  (median prediction error)")
print(f"  RMSE:   ${rmse_eval:>10,.0f}")
print(f"  MAE:    ${mae_eval:>10,.0f}")
print(f"{'='*80}")

r2_diff = r2_train - r2_eval
if r2_diff > 0.1:
    print(f"\n Warning: Possible overfitting detected (R² difference: {r2_diff:.4f})")
else:
    print(f"\n Model generalization looks good (R² difference: {r2_diff:.4f})")

In [ ]:
# Visualizations

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_train, y_pred_train, alpha=0.3, s=20, edgecolor='none')
min_val = min(y_train.min(), y_pred_train.min())
max_val = max(y_train.max(), y_pred_train.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price ($)', fontsize=12)
axes[0].set_ylabel('Predicted Price ($)', fontsize=12)
axes[0].set_title(f'Training Set\nR²={r2_train:.4f}, MdAPE={mdape_train:.2f}%', 
                  fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

axes[1].scatter(y_test_eval, y_pred_eval, alpha=0.5, s=30, edgecolor='none', color='orange')
min_val = min(y_test_eval.min(), y_pred_eval.min())
max_val = max(y_test_eval.max(), y_pred_eval.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Price ($)', fontsize=12)
axes[1].set_ylabel('Predicted Price ($)', fontsize=12)
axes[1].set_title(f'Test Set (0.5% trimmed)\nR²={r2_eval:.4f}, MdAPE={mdape_eval:.2f}%', 
                  fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

for ax in axes:
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))

plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test_eval - y_pred_eval
residuals_pct = (residuals / y_test_eval) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_pred_eval, residuals, alpha=0.5, s=30, edgecolor='none', color='purple')
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Price ($)', fontsize=12)
axes[0].set_ylabel('Residual ($)', fontsize=12)
axes[0].set_title('Residual Plot', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M' if abs(x) >= 1e6 else f'${x/1e3:.0f}K'))

axes[1].hist(residuals_pct, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(x=0, color='r', linestyle='--', lw=2, label='Zero Error')
axes[1].axvline(x=np.median(residuals_pct), color='green', linestyle='--', lw=2, 
                label=f'Median: {np.median(residuals_pct):.2f}%')
axes[1].set_xlabel('Prediction Error (%)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Percentage Errors', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residual Statistics:")
print(f"  Mean residual: ${np.mean(residuals):,.0f}")
print(f"  Median residual: ${np.median(residuals):,.0f}")
print(f"  Std dev of residuals: ${np.std(residuals):,.0f}")

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 20 Most Important Features:")
print("="*60)
for idx, row in feature_importance.head(20).iterrows():
    print(f"{row['Feature']:<30} {row['Importance']:>8.4f}")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

top_15 = feature_importance.head(15)
axes[0].barh(range(len(top_15)), top_15['Importance'].values)
axes[0].set_yticks(range(len(top_15)))
axes[0].set_yticklabels(top_15['Feature'].values)
axes[0].set_xlabel('Importance Score', fontsize=12)
axes[0].set_title('Top 15 Most Important Features', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(alpha=0.3, axis='x')

cumsum_importance = feature_importance['Importance'].cumsum()
axes[1].plot(range(1, len(cumsum_importance)+1), cumsum_importance, linewidth=2, marker='o', markersize=4)
axes[1].axhline(y=0.8, color='r', linestyle='--', lw=2, label='80% threshold')
axes[1].axhline(y=0.9, color='orange', linestyle='--', lw=2, label='90% threshold')
axes[1].set_xlabel('Number of Features', fontsize=12)
axes[1].set_ylabel('Cumulative Importance', fontsize=12)
axes[1].set_title('Cumulative Feature Importance', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

n_80 = (cumsum_importance >= 0.8).argmax() + 1
n_90 = (cumsum_importance >= 0.9).argmax() + 1
print(f"\n✓ {n_80} features account for 80% of total importance")
print(f"✓ {n_90} features account for 90% of total importance")

In [ ]:
n_samples = 10
sample_indices = np.random.choice(len(y_test_eval), n_samples, replace=False)

comparison_df = pd.DataFrame({
    'Actual Price': y_test_eval[sample_indices],
    'Predicted Price': y_pred_eval[sample_indices]
})
comparison_df['Error ($)'] = comparison_df['Actual Price'] - comparison_df['Predicted Price']
comparison_df['Error (%)'] = (comparison_df['Error ($)'] / comparison_df['Actual Price']) * 100
comparison_df['Abs Error (%)'] = comparison_df['Error (%)'].abs()

print(f"{'='*100}")
print(f"SAMPLE PREDICTIONS (Random {n_samples} properties)")
print(f"{'='*100}")
print(comparison_df.to_string(index=False, float_format=lambda x: f'{x:,.2f}' if abs(x) < 100 else f'{x:,.0f}'))
print(f"{'='*100}")

print(f"\nAverage Absolute Error: {comparison_df['Abs Error (%)'].mean():.2f}%")
print(f"Median Absolute Error: {comparison_df['Abs Error (%)'].median():.2f}%")

In [ ]:
error_pct = np.abs((y_test_eval - y_pred_eval) / y_test_eval) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter1 = axes[0].scatter(X_test_eval['Longitude'], X_test_eval['Latitude'],
                          c=error_pct, cmap='RdYlGn_r', alpha=0.6, s=30, 
                          vmin=0, vmax=50, edgecolor='none')
axes[0].set_xlabel('Longitude', fontsize=12)
axes[0].set_ylabel('Latitude', fontsize=12)
axes[0].set_title('Prediction Error by Location', fontsize=14, fontweight='bold')
cbar1 = plt.colorbar(scatter1, ax=axes[0])
cbar1.set_label('Absolute Error (%)', fontsize=11)

hexbin = axes[1].hexbin(X_test_eval['Longitude'], X_test_eval['Latitude'],
                        C=error_pct, gridsize=25, cmap='RdYlGn_r',
                        reduce_C_function=np.median, mincnt=1)
axes[1].set_xlabel('Longitude', fontsize=12)
axes[1].set_ylabel('Latitude', fontsize=12)
axes[1].set_title('Median Prediction Error by Area', fontsize=14, fontweight='bold')
cbar2 = plt.colorbar(hexbin, ax=axes[1])
cbar2.set_label('Median Error (%)', fontsize=11)

plt.tight_layout()
plt.show()